# BirdCLEF 2026 — Ensemble Submission (Top-3 Average)

**Models averaged:**
- `best-derived-v2` (val ROC-AUC 0.9738, Kaggle 0.5142)
- `birdclef25-soundscape` (val 0.9750, Kaggle 0.5131)
- `soundscape-in-train` (val 0.9757, Kaggle 0.5128)

All models share the same architecture: Perch v2 + MLP head (hidden=512, dropout=0.3).
Predictions are averaged before final submission.


In [ ]:
# Ensure TF 2.20+ (required for Perch v2 StableHLO compatibility)
import subprocess, sys

def _tf_version():
    try:
        import tensorflow as tf
        return tuple(int(x) for x in tf.__version__.split(".")[:2])
    except Exception:
        return (0, 0)

if _tf_version() < (2, 20):
    import os
    tf_wheel = "/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"
    tb_wheel = "/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl"
    if os.path.isfile(tb_wheel):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", tb_wheel], check=False)
    if os.path.isfile(tf_wheel):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", tf_wheel], check=False)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow==2.20.0"], check=False)
    if "tensorflow" in sys.modules:
        del sys.modules["tensorflow"]

import tensorflow as tf
print(f"TF version: {tf.__version__}")
assert tuple(int(x) for x in tf.__version__.split(".")[:2]) >= (2, 20), \n    f"TF 2.20+ required for Perch StableHLO, got {tf.__version__}"


In [ ]:
import glob
import os
import re
import time

import librosa
import numpy as np
import pandas as pd
import tensorflow as tf

print('TF version:', tf.__version__)

START_TIME = time.time()
TERMINATE_TIME = START_TIME + 5_400  # 90 min safety cutoff

## Paths — adjust for your Kaggle dataset names

In [ ]:
# ── Data paths ────────────────────────────────────────────────────────────────
DATA_DIR  = '/kaggle/input/birdclef-2026'
PERCH_DIR = '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1'

# Upload each checkpoint as a separate Kaggle dataset
WEIGHTS_PATHS = [
    '/kaggle/input/birdclef2026-best-derived-v2/best_head.weights.h5',
    '/kaggle/input/birdclef2026-birdclef25-soundscape/best_head.weights.h5',
    '/kaggle/input/birdclef2026-soundscape-in-train/best_head.weights.h5',
]

NUM_CLASSES   = 234
HIDDEN_DIM    = 512
DROPOUT       = 0.3
SAMPLE_RATE   = 32_000
CLIP_DURATION = 5
CLIP_SAMPLES  = SAMPLE_RATE * CLIP_DURATION
BATCH_SIZE    = 64
ENABLE_TTA    = True

# Soundscapes
TEST_DIR  = os.path.join(DATA_DIR, 'test_soundscapes')
TRAIN_SC  = os.path.join(DATA_DIR, 'train_soundscapes')
SC_DIR    = TEST_DIR if glob.glob(os.path.join(TEST_DIR, '*.ogg')) else TRAIN_SC
print(f'Soundscapes dir: {SC_DIR}')
print(f'Files: {len(glob.glob(os.path.join(SC_DIR, "*.ogg")))}')


## Model Definition (inline — no external src/ dependency)

In [ ]:
class ClassificationHead(tf.keras.Model):
    """Two-layer MLP: Perch embedding → class logits."""

    def __init__(self, num_classes: int, hidden_dim: int = 512, dropout: float = 0.3):
        super().__init__()
        self.fc1     = tf.keras.layers.Dense(hidden_dim)
        self.act     = tf.keras.layers.Activation('relu')
        self.dropout = tf.keras.layers.Dropout(dropout)
        self.fc2     = tf.keras.layers.Dense(num_classes)

    def call(self, x, training: bool = False):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x, training=training)
        return self.fc2(x)

def load_head(weights_path: str) -> ClassificationHead:
    """Build head, warm up, load weights via h5py."""
    h = ClassificationHead(NUM_CLASSES, HIDDEN_DIM, DROPOUT)
    _ = h(tf.zeros((1, EMB_DIM)), training=False)
    import h5py
    with h5py.File(weights_path, 'r') as wf:
        h.fc1.kernel.assign(wf['fc1']['vars']['0'][:])
        h.fc1.bias.assign(  wf['fc1']['vars']['1'][:])
        h.fc2.kernel.assign(wf['fc2']['vars']['0'][:])
        h.fc2.bias.assign(  wf['fc2']['vars']['1'][:])
    print(f'  Loaded ← {weights_path}')
    return h

# ── Load Perch backbone (shared across all heads) ─────────────────────────────
print('Loading Perch …')
perch  = tf.saved_model.load(PERCH_DIR)
sig    = perch.signatures['serving_default']
_dummy = tf.zeros((1, CLIP_SAMPLES), tf.float32)
EMB_KEY = next((k for k in ('embedding','embeddings','label','logits') if k in sig(inputs=_dummy)),
               next(iter(sig(inputs=_dummy).keys())))
EMB_DIM = int(sig(inputs=_dummy)[EMB_KEY].shape[-1])
print(f'Perch output key={EMB_KEY!r}  dim={EMB_DIM}')

# ── Load all heads ────────────────────────────────────────────────────────────
print(f'Loading {len(WEIGHTS_PATHS)} heads …')
heads = [load_head(wp) for wp in WEIGHTS_PATHS]
print(f'{len(heads)} heads loaded.')


## Species Mapping

In [ ]:
sample_sub     = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
target_species = sample_sub.columns[1:].tolist()   # 234 species codes
print(f'Target species: {len(target_species)}')
print(target_species[:5])

## Inference

In [ ]:
START_TIME     = time.time()
TERMINATE_TIME = START_TIME + 8.5 * 3600  # 8.5h safety margin

@tf.function(input_signature=[tf.TensorSpec([None, CLIP_SAMPLES], tf.float32)])
def _perch_embed(clips):
    return tf.stop_gradient(sig(inputs=clips)[EMB_KEY])

def infer_clips_ensemble(clips: np.ndarray) -> np.ndarray:
    """Run all heads on clips; return mean sigmoid probability."""
    embs = []
    for i in range(0, len(clips), BATCH_SIZE):
        batch = tf.constant(clips[i:i + BATCH_SIZE], dtype=tf.float32)
        embs.append(_perch_embed(batch).numpy())
    emb_arr = np.concatenate(embs, axis=0)  # (n, 1536)

    all_probs = []
    for head in heads:
        logits = head(tf.constant(emb_arr), training=False).numpy()
        all_probs.append(1.0 / (1.0 + np.exp(-logits)))  # sigmoid
    return np.mean(all_probs, axis=0)  # (n, 234)

def process_soundscape(filepath: str):
    ss_id  = re.sub(r'\.ogg$', '', os.path.basename(filepath), flags=re.IGNORECASE)
    audio, _ = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    audio  = audio.astype(np.float32)
    n_segs = len(audio) // CLIP_SAMPLES
    if n_segs == 0:
        return [], np.empty((0, NUM_CLASSES))

    clips   = np.stack([audio[i*CLIP_SAMPLES:(i+1)*CLIP_SAMPLES] for i in range(n_segs)])
    row_ids = [f'{ss_id}_{(i+1)*CLIP_DURATION}' for i in range(n_segs)]
    preds   = infer_clips_ensemble(clips)

    if ENABLE_TTA:
        half = CLIP_SAMPLES // 2
        shifted = audio[half:]
        n_shift = len(shifted) // CLIP_SAMPLES
        if n_shift > 0:
            clips_s = np.stack([shifted[i*CLIP_SAMPLES:(i+1)*CLIP_SAMPLES] for i in range(n_shift)])
            preds_s = infer_clips_ensemble(clips_s)
            n_use   = min(len(preds), len(preds_s))
            preds[:n_use] = 0.5 * preds[:n_use] + 0.5 * preds_s[:n_use]

    return row_ids, preds

ogg_files = sorted(glob.glob(os.path.join(SC_DIR, '*.ogg')))
all_row_ids, all_preds = [], []

for k, fpath in enumerate(ogg_files):
    if time.time() > TERMINATE_TIME:
        print(f'Time limit reached at file {k}/{len(ogg_files)} — stopping early.')
        break
    if k % 100 == 0:
        print(f'[{k+1}/{len(ogg_files)}]  elapsed={( time.time()-START_TIME)/60:.1f}m')
    try:
        row_ids, preds = process_soundscape(fpath)
    except Exception as e:
        print(f'  ERROR {os.path.basename(fpath)}: {e}')
        continue
    if len(row_ids) > 0:
        all_row_ids.extend(row_ids)
        all_preds.append(preds)

print(f'Inference done. {len(all_row_ids)} rows in {(time.time()-START_TIME)/60:.1f}m')


## Build Submission

In [ ]:
if not all_preds:
    raise RuntimeError('No predictions generated. Check soundscapes path and model loading.')

predictions = np.concatenate(all_preds, axis=0)  # (n_rows, 234)

submission = pd.DataFrame(predictions, columns=target_species)
submission.insert(0, 'row_id', all_row_ids)

# Ensure all rows from sample_submission are present (fill missing with 0)
submission = sample_sub[['row_id']].merge(submission, on='row_id', how='left').fillna(0.0)

submission.to_csv('submission.csv', index=False)
print(f'submission.csv saved — {submission.shape[0]} rows × {len(target_species)} species')
print(f'Elapsed: {(time.time()-START_TIME)/60:.1f} min')
submission.head(10)